# **ONLINE SSVEP BCI Analysis Results**

This Notebook requires the user to during the experiment save the server console commands at a `.txt` file (see next cell for further information) which is submitted here for **accuracy results table** display and **confusion matrix** as well.

One file must be saved for gazing at each target during 40s. Each iteration saves the 4 CCA correlations and its maximum, which are crucial for the calculation of the matrix from this Notebook. 

Remember:

| Cell | Target Frequency |
|---|---|
| Eat  | 8.57 Hz |
| Cold | 10.0 Hz |
| SOS  | 12.0 Hz |
| WC   | 15.0 Hz |



> **IMPORTANT! This following code does NOT run in this Notebook.** Add it to the `server.py` module of the **mindaid-ssvep-bci** repository
> so the server console is saved automatically as a `.txt` file (the file you
> later load here).

**In the imports:**

```python
import sys, os
from datetime import datetime
from config import RECORD_DIR   # add it to the config import line at the server.py module

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, d):
        for s in self.streams: s.write(d)
    def flush(self):
        for s in self.streams: s.flush()
```

**At the very start of `main()`:**

```python
os.makedirs(RECORD_DIR, exist_ok=True)
_label = CELLS[TARGET_CELL]["label"]
_ts    = datetime.now().strftime("%Y%m%d_%H%M%S")
_log   = open(os.path.join(RECORD_DIR, f"results_{_label}_{_ts}.txt"),
              "w", encoding="utf-8", buffering=1)
sys.stdout = _Tee(sys.stdout, _log)
```

## ***Initial Configuration***

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display

FOLDER     = "results"            # YOU SHOULD PASTE THE SAVED .txt logs from the server console at a created path with this name for example
FILE_NAME      = "*.txt"
SAVE_PNG = True                   

FREQS = [8.57, 10.0, 12.0, 15.0]
LAB   = ["Eat", "Cold", "SOS", "WC"]
TICK  = ["Eat\n8.57 Hz", "Cold\n10 Hz", "SOS\n12 Hz", "WC\n15 Hz"]

NAME_USER = "Subject 1"
NAME_USER_2 = "Subject 2"
KEY_USER_2  = "name" # you should paste the name you address the user to identify which session corresponds to him/her

print("Configuration loaded.")

## ***Loading the saved file***

In [ ]:
def target_name(fn):
    u = os.path.basename(fn).upper()
    if "EAT"  in u: return 8.57
    if "COLD" in u: return 10.0
    if "SOS"  in u: return 12.0
    if "WC"   in u: return 15.0
    return None

def subject_name(fn):
    return NAME_USER_2 if KEY_USER_2 in os.path.basename(fn).lower() else NAME_USER

cca = re.compile(r"\[CCA\]\s*([\d.]+)Hz:\s*([\d.]+)")
result = re.compile(r"\[Result\].*Corr=([\d.]+)")

def parsing(fn):
    target = target_name(fn)
    subject   = subject_name(fn)
    if target is None:
        print(f"  Warning: target not recognized for name: {os.path.basename(fn)} (discarded)")
        return []
    registrations, cur = [], {}
    with open(fn, encoding="utf-8", errors="ignore") as f:
        for ln in f:
            m = cca.search(ln)
            if m:
                fk = min(FREQS, key=lambda x: abs(x - float(m.group(1))))
                cur[fk] = float(m.group(2))
                continue
            if result.search(ln) and len(cur) == 4:
                rhos = [cur[fr] for fr in FREQS]
                pred = FREQS[int(np.argmax(rhos))]
                registrations.append(dict(subject=subject, target=target, rhos=rhos, pred=pred))
                cur = {}
    return registrations


files = sorted(glob.glob(os.path.join(FOLDER, FILE_NAME)))
assert files, f"Files were not found as {FILE_NAME!r} at {FOLDER!r}"

REG = []
for fn in files:
    r = parsing(fn); REG += r
    print(f"{os.path.basename(fn):45}  {len(r):4d} decisions")
print(f"\nTOTAL: {len(REG)} decisions at {len(files)} files")

subjects = sorted(set(r["subject"] for r in REG))
def subset(subject=None):
    return [r for r in REG if (subject is None or r["subject"] == subject)]
    

## ***Table per recorded session***

In [ ]:
def tables(subject):
    rows = subset(subject)
    print(f"\n {subject}  (N={len(rows)})\n")
    print(f"{'Cells':6}{'Freq':>7}{'Accuracy':>16}{'target_corr':>12}{'winner_corr':>12}")
    for k, fr in enumerate(FREQS):
        rr = [r for r in rows if r["target"] == fr]
        if not rr:
            continue
        corr    = sum(r["pred"] == fr for r in rr)
        target_corr = np.mean([r["rhos"][k]   for r in rr])
        winner_corr = np.mean([max(r["rhos"]) for r in rr])
        print(f"{LAB[k]:6}{fr:7.2f}{f'{corr}/{len(rr)} ({100*corr/len(rr):.1f}%)':>16}"
              f"{target_corr:12.3f}{winner_corr:12.3f}")
    tot = len(rows); acc = sum(r["pred"] == r["target"] for r in rows)
    print(f"{'TOTAL':6}{'':7}{f'{acc}/{tot} ({100*acc/tot:.1f}%)':>16}")

for s in subjects:
    tables(s)

## ***Confusion Matrix***

In [ ]:
def matrix(subject=None):
    M = np.zeros((4, 4), int)
    for r in subset(subject):
        M[FREQS.index(r["target"])][FREQS.index(r["pred"])] += 1
    return M


def plot_matrix(M, title, fname=None):
    valid_rows = M.sum(axis=1) > 0
    rown = np.zeros_like(M, float)
    for i in range(4):
        if M[i].sum() > 0:
            rown[i] = M[i] / M[i].sum()

    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    im = ax.imshow(rown, cmap="Blues", vmin=0, vmax=1)
    for i in range(4):
        if not valid_rows[i]:
            continue
        for j in range(4):
            color = "white" if (i == j or rown[i, j] > 0.55) else "#222"
            ax.text(j, i, f"{M[i,j]}\n{100*rown[i,j]:.1f}%", ha="center", va="center",
                    fontsize=11, color=color, fontweight="bold" if i == j else "normal")
        ax.add_patch(Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor="#E8820C", lw=2.5))

    ax.set_xticks(range(4)); ax.set_xticklabels(TICK, fontsize=9, fontweight="bold")
    ax.set_yticks(range(4)); ax.set_yticklabels([])
    for i, label in enumerate(TICK):
        ax.text(-0.10, i, label, transform=ax.get_yaxis_transform(), ha="center", va="center",
                fontsize=10, fontweight="bold", multialignment="center")
    ax.tick_params(axis="y", pad=10)
    ax.set_xlabel("Predicted symbol", fontsize=10, fontweight="bold", labelpad=15)
    ax.set_ylabel("Target symbol",    fontsize=10, fontweight="bold", labelpad=45)
    N = M.sum(); acc = 100 * np.trace(M) / N if N else 0
    ax.set_title(f"{title}\nAccuracy {acc:.1f}%", fontsize=11, fontweight="bold")
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Classification proportion", fontsize=9)
    fig.tight_layout()
    if fname and SAVE_PNG:
        fig.savefig(fname, dpi=150, bbox_inches="tight"); print("saved as:", fname)
    plt.show()


for s in subjects:
    slug = re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
    plot_matrix(matrix(s), f"Confusion matrix — {s}", fname=f"cm_{slug}.png")

